In [28]:
%%capture
!pip install openpyxl python-docx plotly kaleido pandas numpy matplotlib seaborn

Import Libraries

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from datetime import datetime
from google.colab import files
import io
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


Upload Your Files

In [30]:
print("📁 Select your Excel/CSV files for analysis")
print("You can select multiple files at once!\n")

uploaded = files.upload()

print(f"\n✅ {len(uploaded)} file(s) uploaded successfully!")
for filename in uploaded.keys():
    print(f"  - {filename}")

📁 Select your Excel/CSV files for analysis
You can select multiple files at once!



Saving Activités par Unité et Saison.xlsx to Activités par Unité et Saison (2).xlsx
Saving Activites_Generales.xlsx to Activites_Generales (2).xlsx
Saving american-bsa_merit_badges (1).xlsx to american-bsa_merit_badges (1) (2).xlsx
Saving american-bsa_merit_badges.xlsx to american-bsa_merit_badges (3).xlsx
Saving american-scout-partners (1).xlsx to american-scout-partners (1) (2).xlsx
Saving american-scout-partners.xlsx to american-scout-partners (3).xlsx
Saving Budgets par Type d'Activité.xlsx to Budgets par Type d'Activité (2).xlsx
Saving Budgets_et_Finances.xlsx to Budgets_et_Finances (2).xlsx
Saving Camps_Detailles.xlsx to Camps_Detailles (2).xlsx
Saving Classeur1.xlsx to Classeur1 (2).xlsx
Saving Détails des Coûts de Camp (Dinars).xlsx to Détails des Coûts de Camp (Dinars) (2).xlsx
Saving event-france_final (1).xlsx to event-france_final (1) (2).xlsx
Saving event-spain (1).xlsx to event-spain (1) (2).xlsx
Saving Membres par Unité et Saison.xlsx to Membres par Unité et Saison (2).x

Load and Process Files

In [31]:
def read_file(filename, file_content):
    """Read file handling different formats and encodings"""
    try:
        if filename.endswith('.csv'):
            # Try different encodings for CSV
            for encoding in ['utf-8', 'latin1', 'cp1252']:
                try:
                    df = pd.read_csv(io.BytesIO(file_content), encoding=encoding)
                    return {'Sheet1': df}
                except:
                    continue
            # If CSV fails, try as Excel
            try:
                return pd.read_excel(io.BytesIO(file_content), sheet_name=None)
            except:
                return None
        else:
            return pd.read_excel(io.BytesIO(file_content), sheet_name=None)
    except Exception as e:
        print(f"❌ Error reading {filename}: {e}")
        return None

# Load all uploaded files
dataframes = {}

print("Loading files...\n")
for filename, content in uploaded.items():
    print(f"📊 Loading: {filename}")
    df_dict = read_file(filename, content)

    if df_dict:
        dataframes[filename] = df_dict
        for sheet_name, df in df_dict.items():
            print(f"  ✅ Sheet '{sheet_name}': {len(df)} rows × {len(df.columns)} columns")
    else:
        print(f"  ❌ Failed to load {filename}")

print(f"\n✅ Successfully loaded {len(dataframes)} file(s)")

Loading files...

📊 Loading: Activités par Unité et Saison (2).xlsx
  ✅ Sheet 'Sheet1': 6 rows × 4 columns
📊 Loading: Activites_Generales (2).xlsx
  ✅ Sheet 'Sheet1': 4 rows × 4 columns
📊 Loading: american-bsa_merit_badges (1) (2).xlsx
  ✅ Sheet 'Sheet1': 141 rows × 3 columns
📊 Loading: american-bsa_merit_badges (3).xlsx
  ✅ Sheet 'Sheet1': 141 rows × 3 columns
📊 Loading: american-scout-partners (1) (2).xlsx
  ✅ Sheet 'Sheet1': 38 rows × 2 columns
📊 Loading: american-scout-partners (3).xlsx
  ✅ Sheet 'Sheet1': 38 rows × 2 columns
📊 Loading: Budgets par Type d'Activité (2).xlsx
  ✅ Sheet 'Sheet1': 4 rows × 4 columns
📊 Loading: Budgets_et_Finances (2).xlsx
  ✅ Sheet 'Sheet1': 6 rows × 4 columns
📊 Loading: Camps_Detailles (2).xlsx
  ✅ Sheet 'Sheet1': 7 rows × 5 columns
📊 Loading: Classeur1 (2).xlsx
  ✅ Sheet 'Feuil1': 5 rows × 3 columns
📊 Loading: Détails des Coûts de Camp (Dinars) (2).xlsx
  ✅ Sheet 'Sheet1': 7 rows × 7 columns
📊 Loading: event-france_final (1) (2).xlsx
  ✅ Sheet 'Sheet1

Perform Comprehensive Analysis

In [32]:
def analyze_dataframe(df, file_name, sheet_name):
    """Perform comprehensive EDA and quality analysis"""

    analysis = {
        'file': file_name,
        'sheet': sheet_name,
        'dimensions': {
            'rows': len(df),
            'columns': len(df.columns)
        },
        'columns': {},
        'quality': {},
        'visualizations': []
    }

    # Column-level analysis
    for col in df.columns:
        col_info = {
            'dtype': str(df[col].dtype),
            'non_null': int(df[col].notna().sum()),
            'null': int(df[col].isna().sum()),
            'null_pct': float(df[col].isna().sum() / len(df) * 100) if len(df) > 0 else 0,
            'unique': int(df[col].nunique()),
            'unique_pct': float(df[col].nunique() / len(df) * 100) if len(df) > 0 else 0
        }

        # Sample values
        non_null = df[col].dropna()
        if len(non_null) > 0:
            col_info['samples'] = [str(x) for x in non_null.head(3).tolist()]

        # Statistics for numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            stats = df[col].describe()
            col_info['stats'] = {
                'mean': float(stats['mean']) if pd.notna(stats['mean']) else None,
                'std': float(stats['std']) if pd.notna(stats['std']) else None,
                'min': float(stats['min']) if pd.notna(stats['min']) else None,
                'q25': float(stats['25%']) if pd.notna(stats['25%']) else None,
                'median': float(stats['50%']) if pd.notna(stats['50%']) else None,
                'q75': float(stats['75%']) if pd.notna(stats['75%']) else None,
                'max': float(stats['max']) if pd.notna(stats['max']) else None
            }

        analysis['columns'][col] = col_info

    # Quality metrics
    total_cells = len(df) * len(df.columns)
    missing_cells = df.isna().sum().sum()

    analysis['quality'] = {
        'total_cells': int(total_cells),
        'missing_cells': int(missing_cells),
        'completeness_pct': float((total_cells - missing_cells) / total_cells * 100) if total_cells > 0 else 0,
        'duplicates': int(df.duplicated().sum()),
        'duplicate_pct': float(df.duplicated().sum() / len(df) * 100) if len(df) > 0 else 0,
        'empty_columns': [col for col in df.columns if df[col].isna().all()],
        'complete_columns': [col for col in df.columns if df[col].notna().all()]
    }

    return analysis, df

# Analyze all dataframes
all_analyses = []

print("Performing comprehensive analysis...\n")
for file_name, sheets in dataframes.items():
    print(f"📊 Analyzing: {file_name}")
    for sheet_name, df in sheets.items():
        print(f"  Sheet: {sheet_name}")
        analysis, _ = analyze_dataframe(df, file_name, sheet_name)
        all_analyses.append(analysis)
        print(f"  ✅ Quality Score: {analysis['quality']['completeness_pct']:.1f}%")

print(f"\n✅ Analysis complete for {len(all_analyses)} dataset(s)!")

Performing comprehensive analysis...

📊 Analyzing: Activités par Unité et Saison (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Activites_Generales (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-bsa_merit_badges (1) (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-bsa_merit_badges (3).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-scout-partners (1) (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: american-scout-partners (3).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Budgets par Type d'Activité (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Budgets_et_Finances (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Camps_Detailles (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score: 100.0%
📊 Analyzing: Classeur1 (2).xlsx
  Sheet: Feuil1
  ✅ Quality Score: 100.0%
📊 Analyzing: Détails des Coûts de Camp (Dinars) (2).xlsx
  Sheet: Sheet1
  ✅ Quality Score

Generate Visualizations

In [33]:
def create_visualizations(df, file_name, sheet_name):
    """Create comprehensive visualizations"""

    plot_images = []

    print(f"\n📊 Generating visualizations for {file_name} - {sheet_name}")

    # 1. Missing Data Heatmap
    if df.shape[0] > 0 and df.isnull().any().any():
        fig, ax = plt.subplots(figsize=(12, 6))
        sns.heatmap(df.isnull(), yticklabels=False, cbar=True, cmap='RdYlGn_r', ax=ax)
        ax.set_title(f'Missing Data Pattern - {sheet_name}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Columns')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Missing Data Heatmap', img_buffer))
        plt.close()
        print("  ✅ Missing data heatmap")

    # 2. Missing Data Percentage
    missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
    if missing_pct.sum() > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        missing_pct[missing_pct > 0].plot(kind='bar', ax=ax, color='coral', edgecolor='black')
        ax.set_title(f'Missing Data % by Column - {sheet_name}', fontsize=14, fontweight='bold')
        ax.set_ylabel('Missing %')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Missing Percentage', img_buffer))
        plt.close()
        print("  ✅ Missing percentage chart")

    # 3. Numeric Distributions
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols and len(df) > 0:
        n_cols = min(3, len(numeric_cols))
        fig, axes = plt.subplots(1, n_cols, figsize=(15, 4))
        if n_cols == 1:
            axes = [axes]

        for i, col in enumerate(numeric_cols[:n_cols]):
            data = df[col].dropna()
            if len(data) > 0:
                axes[i].hist(data, bins=30, edgecolor='black', color='steelblue', alpha=0.7)
                axes[i].set_title(f'{col}', fontweight='bold')
                axes[i].set_ylabel('Frequency')

        plt.suptitle('Numeric Distributions', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Numeric Distributions', img_buffer))
        plt.close()
        print("  ✅ Numeric distributions")

    # 4. Categorical Distribution
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    if categorical_cols and len(df) > 0:
        col = categorical_cols[0]
        value_counts = df[col].value_counts().head(10)

        if len(value_counts) > 0:
            fig, ax = plt.subplots(figsize=(12, 6))
            value_counts.plot(kind='barh', ax=ax, color='lightgreen', edgecolor='black')
            ax.set_title(f'{col} - Top 10 Values', fontsize=14, fontweight='bold')
            ax.set_xlabel('Count')
            plt.tight_layout()

            img_buffer = io.BytesIO()
            plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
            img_buffer.seek(0)
            plot_images.append(('Categorical Distribution', img_buffer))
            plt.close()
            print("  ✅ Categorical distribution")

    # 5. Correlation Matrix
    if len(numeric_cols) >= 2:
        fig, ax = plt.subplots(figsize=(10, 8))
        corr_matrix = df[numeric_cols].corr()
        sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                    square=True, linewidths=1, ax=ax)
        ax.set_title(f'Correlation Matrix - {sheet_name}', fontsize=14, fontweight='bold')
        plt.tight_layout()

        img_buffer = io.BytesIO()
        plt.savefig(img_buffer, format='png', dpi=150, bbox_inches='tight')
        img_buffer.seek(0)
        plot_images.append(('Correlation Matrix', img_buffer))
        plt.close()
        print("  ✅ Correlation matrix")

    return plot_images

# Generate visualizations for all datasets
all_visualizations = {}

for file_name, sheets in dataframes.items():
    all_visualizations[file_name] = {}
    for sheet_name, df in sheets.items():
        viz = create_visualizations(df, file_name, sheet_name)
        all_visualizations[file_name][sheet_name] = viz

print("\n✅ All visualizations generated!")


📊 Generating visualizations for Activités par Unité et Saison (2).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for Activites_Generales (2).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for american-bsa_merit_badges (1) (2).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution

📊 Generating visualizations for american-bsa_merit_badges (3).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution

📊 Generating visualizations for american-scout-partners (1) (2).xlsx - Sheet1
  ✅ Categorical distribution

📊 Generating visualizations for american-scout-partners (3).xlsx - Sheet1
  ✅ Categorical distribution

📊 Generating visualizations for Budgets par Type d'Activité (2).xlsx - Sheet1
  ✅ Numeric distributions
  ✅ Categorical distribution
  ✅ Correlation matrix

📊 Generating visualizations for Budgets_et_Finances (2

 Generate EDA Word Document

In [34]:
def create_eda_word_report(analyses, visualizations, dataframes):
    """Create EDA Word document report"""

    doc = Document()

    # Title Page
    title = doc.add_heading('EXPLORATORY DATA ANALYSIS REPORT', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    subtitle = doc.add_heading('Comprehensive Statistical Analysis & Data Exploration', level=1)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

    date_para = doc.add_paragraph(f"Generated: {datetime.now().strftime('%B %d, %Y at %H:%M')}")
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()

    # Executive Summary
    doc.add_heading('EXECUTIVE SUMMARY', 1)
    doc.add_paragraph(
        'This Exploratory Data Analysis (EDA) report provides a comprehensive statistical analysis '
        'of all uploaded datasets. The report includes descriptive statistics, data distributions, '
        'correlations, and detailed visualizations for each dataset.'
    )

    doc.add_heading('Analysis Overview', 2)
    doc.add_paragraph(f"Total Files Analyzed: {len(dataframes)}", style='List Bullet')
    doc.add_paragraph(f"Total Datasets: {len(analyses)}", style='List Bullet')
    total_rows = sum(a['dimensions']['rows'] for a in analyses)
    doc.add_paragraph(f"Total Rows: {total_rows:,}", style='List Bullet')

    doc.add_page_break()

    # Individual File Analyses
    for idx, analysis in enumerate(analyses, 1):
        doc.add_heading(f"{idx}. {analysis['file']} - {analysis['sheet']}", 1)

        # Dataset Overview
        doc.add_heading('Dataset Overview', 2)
        doc.add_paragraph(f"Rows: {analysis['dimensions']['rows']:,}", style='List Bullet')
        doc.add_paragraph(f"Columns: {analysis['dimensions']['columns']}", style='List Bullet')

        # Column Details Table
        doc.add_heading('Column Analysis', 2)
        table = doc.add_table(rows=1, cols=6)
        table.style = 'Light Grid Accent 1'

        header_cells = table.rows[0].cells
        header_cells[0].text = 'Column'
        header_cells[1].text = 'Type'
        header_cells[2].text = 'Non-Null'
        header_cells[3].text = 'Null %'
        header_cells[4].text = 'Unique'
        header_cells[5].text = 'Unique %'

        for col, col_data in list(analysis['columns'].items())[:15]:
            row_cells = table.add_row().cells
            row_cells[0].text = col
            row_cells[1].text = col_data['dtype']
            row_cells[2].text = f"{col_data['non_null']:,}"
            row_cells[3].text = f"{col_data['null_pct']:.1f}%"
            row_cells[4].text = f"{col_data['unique']:,}"
            row_cells[5].text = f"{col_data['unique_pct']:.1f}%"

        if len(analysis['columns']) > 15:
            doc.add_paragraph(f"... and {len(analysis['columns']) - 15} more columns", style='Intense Quote')

        # Statistical Summary
        doc.add_heading('Statistical Summary', 2)

        # Create statistics table for numeric columns
        numeric_cols = {col: data for col, data in analysis['columns'].items() if 'stats' in data}

        if numeric_cols:
            stats_table = doc.add_table(rows=1, cols=8)
            stats_table.style = 'Light Grid Accent 1'

            stat_headers = stats_table.rows[0].cells
            stat_headers[0].text = 'Column'
            stat_headers[1].text = 'Mean'
            stat_headers[2].text = 'Std Dev'
            stat_headers[3].text = 'Min'
            stat_headers[4].text = '25%'
            stat_headers[5].text = 'Median'
            stat_headers[6].text = '75%'
            stat_headers[7].text = 'Max'

            for col, col_data in list(numeric_cols.items())[:10]:
                row_cells = stats_table.add_row().cells
                stats = col_data['stats']
                row_cells[0].text = col
                row_cells[1].text = f"{stats.get('mean', 'N/A'):.2f}" if stats.get('mean') else 'N/A'
                row_cells[2].text = f"{stats.get('std', 'N/A'):.2f}" if stats.get('std') else 'N/A'
                row_cells[3].text = f"{stats.get('min', 'N/A'):.2f}" if stats.get('min') else 'N/A'
                row_cells[4].text = f"{stats.get('q25', 'N/A'):.2f}" if stats.get('q25') else 'N/A'
                row_cells[5].text = f"{stats.get('median', 'N/A'):.2f}" if stats.get('median') else 'N/A'
                row_cells[6].text = f"{stats.get('q75', 'N/A'):.2f}" if stats.get('q75') else 'N/A'
                row_cells[7].text = f"{stats.get('max', 'N/A'):.2f}" if stats.get('max') else 'N/A'
        else:
            doc.add_paragraph('No numeric columns found in this dataset.', style='Intense Quote')

        # Data Visualizations
        doc.add_heading('Data Visualizations', 2)

        if analysis['file'] in visualizations and analysis['sheet'] in visualizations[analysis['file']]:
            plots = visualizations[analysis['file']][analysis['sheet']]
            for plot_name, plot_buffer in plots:
                doc.add_heading(plot_name, 3)
                plot_buffer.seek(0)
                doc.add_picture(plot_buffer, width=Inches(6))
                doc.add_paragraph()
        else:
            doc.add_paragraph('No visualizations available for this dataset.', style='Intense Quote')

        doc.add_page_break()

    # Overall Summary
    doc.add_heading('OVERALL SUMMARY', 1)

    total_cols = sum(a['dimensions']['columns'] for a in analyses)
    avg_rows = total_rows / len(analyses) if analyses else 0

    doc.add_heading('Dataset Statistics', 2)
    doc.add_paragraph(f"Total Datasets: {len(analyses)}", style='List Bullet')
    doc.add_paragraph(f"Total Rows Analyzed: {total_rows:,}", style='List Bullet')
    doc.add_paragraph(f"Total Columns Analyzed: {total_cols}", style='List Bullet')
    doc.add_paragraph(f"Average Rows per Dataset: {avg_rows:,.0f}", style='List Bullet')

    return doc

print("📄 Generating EDA Word document...")
eda_doc = create_eda_word_report(all_analyses, all_visualizations, dataframes)
eda_word_filename = f"EDA_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"
eda_doc.save(eda_word_filename)
print(f"✅ EDA Word report created: {eda_word_filename}")

📄 Generating EDA Word document...
✅ EDA Word report created: EDA_Report_20260201_2000.docx


Generate Data Quality Word Document

In [35]:
def create_quality_word_report(analyses, dataframes):
    """Create Data Quality Word document report"""

    doc = Document()

    # Title Page
    title = doc.add_heading('DATA QUALITY ASSESSMENT REPORT', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    subtitle = doc.add_heading('Comprehensive Data Quality Analysis & Recommendations', level=1)
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

    date_para = doc.add_paragraph(f"Generated: {datetime.now().strftime('%B %d, %Y at %H:%M')}")
    date_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()

    # Executive Summary
    doc.add_heading('EXECUTIVE SUMMARY', 1)
    doc.add_paragraph(
        'This Data Quality Assessment report evaluates the quality, completeness, and integrity '
        'of all uploaded datasets. The report includes quality scores, issue identification, '
        'and actionable recommendations for data improvement.'
    )

    # Overall Quality Metrics
    avg_completeness = sum(a['quality']['completeness_pct'] for a in analyses) / len(analyses)
    total_duplicates = sum(a['quality']['duplicates'] for a in analyses)
    total_missing = sum(a['quality']['missing_cells'] for a in analyses)

    doc.add_heading('Overall Quality Metrics', 2)
    doc.add_paragraph(f"Average Completeness: {avg_completeness:.2f}%", style='List Bullet')
    doc.add_paragraph(f"Total Missing Cells: {total_missing:,}", style='List Bullet')
    doc.add_paragraph(f"Total Duplicate Rows: {total_duplicates:,}", style='List Bullet')

    doc.add_page_break()

    # Individual File Quality Assessments
    for idx, analysis in enumerate(analyses, 1):
        doc.add_heading(f"{idx}. {analysis['file']} - {analysis['sheet']}", 1)

        quality = analysis['quality']

        # Quality Score Card
        doc.add_heading('Quality Score Card', 2)

        completeness = quality['completeness_pct']
        if completeness >= 95:
            rating = "EXCELLENT"
            score = 95
            color = RGBColor(0, 170, 0)
        elif completeness >= 80:
            rating = "GOOD"
            score = 80
            color = RGBColor(255, 165, 0)
        elif completeness >= 60:
            rating = "FAIR"
            score = 65
            color = RGBColor(255, 136, 0)
        else:
            rating = "NEEDS IMPROVEMENT"
            score = 40
            color = RGBColor(255, 0, 0)

        quality_para = doc.add_paragraph(f"Quality Rating: {rating}")
        quality_para.runs[0].font.bold = True
        quality_para.runs[0].font.size = Pt(16)
        quality_para.runs[0].font.color.rgb = color

        score_para = doc.add_paragraph(f"Quality Score: {score}/100")
        score_para.runs[0].font.bold = True
        score_para.runs[0].font.size = Pt(14)

        # Detailed Quality Metrics
        doc.add_heading('Quality Metrics', 2)

        metrics_table = doc.add_table(rows=5, cols=2)
        metrics_table.style = 'Light Grid Accent 1'

        metrics_table.rows[0].cells[0].text = 'Metric'
        metrics_table.rows[0].cells[1].text = 'Value'

        metrics_table.rows[1].cells[0].text = 'Data Completeness'
        metrics_table.rows[1].cells[1].text = f"{quality['completeness_pct']:.2f}%"

        metrics_table.rows[2].cells[0].text = 'Missing Cells'
        metrics_table.rows[2].cells[1].text = f"{quality['missing_cells']:,}"

        metrics_table.rows[3].cells[0].text = 'Duplicate Rows'
        metrics_table.rows[3].cells[1].text = f"{quality['duplicates']:,} ({quality['duplicate_pct']:.2f}%)"

        metrics_table.rows[4].cells[0].text = 'Empty Columns'
        metrics_table.rows[4].cells[1].text = str(len(quality['empty_columns']))

        # Quality Checks
        doc.add_heading('Quality Checks', 2)

        # Completeness Check
        doc.add_heading('1. Completeness Check', 3)
        if completeness >= 95:
            doc.add_paragraph(f"✅ EXCELLENT: {completeness:.2f}% complete", style='List Bullet')
        elif completeness >= 80:
            doc.add_paragraph(f"⚠️ GOOD: {completeness:.2f}% complete", style='List Bullet')
        elif completeness >= 60:
            doc.add_paragraph(f"⚠️ FAIR: {completeness:.2f}% complete", style='List Bullet')
        else:
            doc.add_paragraph(f"❌ POOR: {completeness:.2f}% complete", style='List Bullet')

        # Duplicate Check
        doc.add_heading('2. Duplicate Check', 3)
        if quality['duplicates'] == 0:
            doc.add_paragraph("✅ NO DUPLICATES FOUND", style='List Bullet')
        else:
            doc.add_paragraph(f"⚠️ {quality['duplicates']:,} duplicate rows found ({quality['duplicate_pct']:.2f}%)", style='List Bullet')

        # Empty Columns Check
        doc.add_heading('3. Empty Columns Check', 3)
        if quality['empty_columns']:
            doc.add_paragraph(f"⚠️ Columns with ALL null values:", style='List Bullet')
            for col in quality['empty_columns']:
                doc.add_paragraph(f"  • {col}", style='List Bullet 2')
        else:
            doc.add_paragraph("✅ NO COMPLETELY EMPTY COLUMNS", style='List Bullet')

        # High Null Columns
        doc.add_heading('4. High Null Columns (>50%)', 3)
        high_null_cols = [(col, data['null_pct']) for col, data in analysis['columns'].items() if data['null_pct'] > 50]

        if high_null_cols:
            doc.add_paragraph(f"⚠️ Found {len(high_null_cols)} columns with >50% null values:", style='List Bullet')
            for col, pct in high_null_cols[:10]:
                doc.add_paragraph(f"  • {col}: {pct:.1f}% null", style='List Bullet 2')
        else:
            doc.add_paragraph("✅ NO COLUMNS WITH >50% NULL VALUES", style='List Bullet')

        # Issues & Recommendations
        doc.add_heading('Issues & Recommendations', 2)

        issues_found = False

        if completeness < 95:
            issues_found = True
            doc.add_paragraph(f"Issue: Data completeness is {completeness:.2f}%", style='List Bullet')
            doc.add_paragraph("Recommendation: Investigate missing data patterns and implement validation rules", style='List Bullet 2')

        if quality['duplicates'] > 0:
            issues_found = True
            doc.add_paragraph(f"Issue: Found {quality['duplicates']:,} duplicate rows", style='List Bullet')
            doc.add_paragraph("Recommendation: Review and remove duplicate records to ensure data integrity", style='List Bullet 2')

        if quality['empty_columns']:
            issues_found = True
            doc.add_paragraph(f"Issue: {len(quality['empty_columns'])} empty columns provide no value", style='List Bullet')
            doc.add_paragraph("Recommendation: Remove empty columns from the dataset", style='List Bullet 2')

        if high_null_cols:
            issues_found = True
            doc.add_paragraph(f"Issue: {len(high_null_cols)} columns with >50% null values", style='List Bullet')
            doc.add_paragraph("Recommendation: Review if these columns are necessary or can be filled with default values", style='List Bullet 2')

        if not issues_found:
            doc.add_paragraph("✅ No major quality issues identified!", style='List Bullet')

        doc.add_page_break()

    # Overall Recommendations
    doc.add_heading('OVERALL RECOMMENDATIONS', 1)

    doc.add_heading('High Priority Actions', 2)
    doc.add_paragraph('Address datasets with completeness below 80%', style='List Bullet')
    doc.add_paragraph('Remove or investigate all duplicate records', style='List Bullet')
    doc.add_paragraph('Clean up empty columns that provide no analytical value', style='List Bullet')

    doc.add_heading('Medium Priority Actions', 2)
    doc.add_paragraph('Review columns with high null percentages (>50%)', style='List Bullet')
    doc.add_paragraph('Implement data validation rules at data entry points', style='List Bullet')
    doc.add_paragraph('Document data collection and validation procedures', style='List Bullet')

    doc.add_heading('Long-term Improvements', 2)
    doc.add_paragraph('Establish automated data quality monitoring', style='List Bullet')
    doc.add_paragraph('Create data quality dashboards for ongoing tracking', style='List Bullet')
    doc.add_paragraph('Implement regular data quality audits', style='List Bullet')
    doc.add_paragraph('Train data entry personnel on quality standards', style='List Bullet')

    return doc

print("📋 Generating Data Quality Word document...")
quality_doc = create_quality_word_report(all_analyses, dataframes)
quality_word_filename = f"Data_Quality_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.docx"
quality_doc.save(quality_word_filename)
print(f"✅ Quality Word report created: {quality_word_filename}")

📋 Generating Data Quality Word document...
✅ Quality Word report created: Data_Quality_Report_20260201_2000.docx


Generate Excel Summary

In [36]:
def create_excel_summary(analyses):
    """Create Excel summary of all analyses"""

    summary_data = []

    for analysis in analyses:
        summary_data.append({
            'File': analysis['file'],
            'Sheet': analysis['sheet'],
            'Rows': analysis['dimensions']['rows'],
            'Columns': analysis['dimensions']['columns'],
            'Completeness %': round(analysis['quality']['completeness_pct'], 2),
            'Missing Cells': analysis['quality']['missing_cells'],
            'Duplicate Rows': analysis['quality']['duplicates'],
            'Empty Columns': len(analysis['quality']['empty_columns'])
        })

    df_summary = pd.DataFrame(summary_data)

    excel_filename = f"Analysis_Summary_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        df_summary.to_excel(writer, sheet_name='Summary', index=False)

        # Format the sheet
        workbook = writer.book
        worksheet = writer.sheets['Summary']

        from openpyxl.styles import Font, PatternFill, Alignment

        # Header formatting
        header_fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        header_font = Font(bold=True, color='FFFFFF')

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center')

        # Auto-adjust column widths
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            worksheet.column_dimensions[column_letter].width = adjusted_width

    return excel_filename

print("📊 Generating Excel summary...")
excel_filename = create_excel_summary(all_analyses)
print(f"✅ Excel summary created: {excel_filename}")

📊 Generating Excel summary...
✅ Excel summary created: Analysis_Summary_20260201_2000.xlsx


Generate Text Reports

In [37]:
def generate_text_reports(analyses):
    """Generate comprehensive text reports"""

    # EDA Report
    eda_lines = []
    eda_lines.append("=" * 100)
    eda_lines.append("EXPLORATORY DATA ANALYSIS (EDA) REPORT")
    eda_lines.append("=" * 100)
    eda_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    eda_lines.append(f"Total Datasets Analyzed: {len(analyses)}")
    eda_lines.append("")

    for analysis in analyses:
        eda_lines.append("\n" + "=" * 100)
        eda_lines.append(f"FILE: {analysis['file']} - SHEET: {analysis['sheet']}")
        eda_lines.append("=" * 100)

        eda_lines.append("\n1. DATASET DIMENSIONS:")
        eda_lines.append(f"   • Rows: {analysis['dimensions']['rows']:,}")
        eda_lines.append(f"   • Columns: {analysis['dimensions']['columns']}")

        eda_lines.append("\n2. COLUMN ANALYSIS:")
        for col, col_data in analysis['columns'].items():
            eda_lines.append(f"\n   Column: {col}")
            eda_lines.append(f"   ├─ Type: {col_data['dtype']}")
            eda_lines.append(f"   ├─ Non-Null: {col_data['non_null']:,} ({100 - col_data['null_pct']:.1f}%)")
            eda_lines.append(f"   ├─ Null: {col_data['null']:,} ({col_data['null_pct']:.1f}%)")
            eda_lines.append(f"   └─ Unique: {col_data['unique']:,} ({col_data['unique_pct']:.1f}%)")

            if 'stats' in col_data and col_data['stats']:
                stats = col_data['stats']
                eda_lines.append(f"      Statistics: Mean={stats.get('mean', 'N/A')}, Median={stats.get('median', 'N/A')}, Std={stats.get('std', 'N/A')}")

    # Quality Report
    quality_lines = []
    quality_lines.append("=" * 100)
    quality_lines.append("DATA QUALITY REPORT")
    quality_lines.append("=" * 100)
    quality_lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    quality_lines.append("")

    for analysis in analyses:
        quality_lines.append("\n" + "=" * 100)
        quality_lines.append(f"FILE: {analysis['file']} - SHEET: {analysis['sheet']}")
        quality_lines.append("=" * 100)

        quality = analysis['quality']

        quality_lines.append("\n1. COMPLETENESS CHECK:")
        completeness = quality['completeness_pct']
        if completeness >= 95:
            quality_lines.append(f"   ✅ EXCELLENT: {completeness:.2f}% complete")
        elif completeness >= 80:
            quality_lines.append(f"   ⚠️  GOOD: {completeness:.2f}% complete")
        elif completeness >= 60:
            quality_lines.append(f"   ⚠️  FAIR: {completeness:.2f}% complete")
        else:
            quality_lines.append(f"   ❌ POOR: {completeness:.2f}% complete")

        quality_lines.append("\n2. DUPLICATE CHECK:")
        if quality['duplicates'] == 0:
            quality_lines.append("   ✅ NO DUPLICATES FOUND")
        else:
            quality_lines.append(f"   ⚠️  {quality['duplicates']:,} duplicate rows ({quality['duplicate_pct']:.2f}%)")

        quality_lines.append("\n3. EMPTY COLUMNS:")
        if quality['empty_columns']:
            quality_lines.append(f"   ⚠️  Columns with ALL nulls: {', '.join(quality['empty_columns'])}")
        else:
            quality_lines.append("   ✅ NO COMPLETELY EMPTY COLUMNS")

    # Save reports
    eda_filename = f"EDA_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"
    quality_filename = f"Quality_Report_{datetime.now().strftime('%Y%m%d_%H%M')}.txt"

    with open(eda_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(eda_lines))

    with open(quality_filename, 'w', encoding='utf-8') as f:
        f.write('\n'.join(quality_lines))

    return eda_filename, quality_filename

print("📝 Generating text reports...")
eda_txt, quality_txt = generate_text_reports(all_analyses)
print(f"✅ EDA text report: {eda_txt}")
print(f"✅ Quality text report: {quality_txt}")

📝 Generating text reports...
✅ EDA text report: EDA_Report_20260201_2000.txt
✅ Quality text report: Quality_Report_20260201_2000.txt


Download All Reports

In [38]:
print("📥 Downloading all reports...\n")

# Download EDA Word report
print(f"📄 Downloading: {eda_word_filename}")
files.download(eda_word_filename)

# Download Quality Word report
print(f"📋 Downloading: {quality_word_filename}")
files.download(quality_word_filename)

# Download Excel summary
print(f"📊 Downloading: {excel_filename}")
files.download(excel_filename)

# Download text reports
print(f"📝 Downloading: {eda_txt}")
files.download(eda_txt)

print(f"📝 Downloading: {quality_txt}")
files.download(quality_txt)

print("\n✅ All reports downloaded successfully!")
print("\n📦 Generated Reports:")
print(f"  1. {eda_word_filename} - EDA Word Document")
print(f"  2. {quality_word_filename} - Quality Word Document")
print(f"  3. {excel_filename} - Excel Summary")
print(f"  4. {eda_txt} - EDA Text Report")
print(f"  5. {quality_txt} - Quality Text Report")

📥 Downloading all reports...

📄 Downloading: EDA_Report_20260201_2000.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📋 Downloading: Data_Quality_Report_20260201_2000.docx


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

📝 Downloading: Quality_Report_20260201_2000.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All reports downloaded successfully!

📦 Generated Reports:
  1. EDA_Report_20260201_2000.docx - EDA Word Document
  2. Data_Quality_Report_20260201_2000.docx - Quality Word Document
  3. Analysis_Summary_20260201_2000.xlsx - Excel Summary
  4. EDA_Report_20260201_2000.txt - EDA Text Report
  5. Quality_Report_20260201_2000.txt - Quality Text Report


Analysis Summary

In [39]:
print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)

for analysis in all_analyses:
    print(f"\n📊 {analysis['file']} - {analysis['sheet']}")
    print(f"   Dimensions: {analysis['dimensions']['rows']:,} rows × {analysis['dimensions']['columns']} columns")
    print(f"   Completeness: {analysis['quality']['completeness_pct']:.2f}%")
    print(f"   Duplicates: {analysis['quality']['duplicates']:,}")

    # Quality rating
    completeness = analysis['quality']['completeness_pct']
    if completeness >= 95:
        rating = "✅ EXCELLENT"
    elif completeness >= 80:
        rating = "⚠️  GOOD"
    elif completeness >= 60:
        rating = "⚠️  FAIR"
    else:
        rating = "❌ NEEDS IMPROVEMENT"
    print(f"   Quality: {rating}")

print("\n" + "="*80)
print(f"Total Datasets: {len(all_analyses)}")
avg_quality = sum(a['quality']['completeness_pct'] for a in all_analyses) / len(all_analyses)
print(f"Average Quality: {avg_quality:.2f}%")
print("="*80)


ANALYSIS SUMMARY

📊 Activités par Unité et Saison (2).xlsx - Sheet1
   Dimensions: 6 rows × 4 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 Activites_Generales (2).xlsx - Sheet1
   Dimensions: 4 rows × 4 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-bsa_merit_badges (1) (2).xlsx - Sheet1
   Dimensions: 141 rows × 3 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-bsa_merit_badges (3).xlsx - Sheet1
   Dimensions: 141 rows × 3 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-scout-partners (1) (2).xlsx - Sheet1
   Dimensions: 38 rows × 2 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 american-scout-partners (3).xlsx - Sheet1
   Dimensions: 38 rows × 2 columns
   Completeness: 100.00%
   Duplicates: 0
   Quality: ✅ EXCELLENT

📊 Budgets par Type d'Activité (2).xlsx - Sheet1
   Dimensions: 4 rows × 4 columns
   Comple

importation des fichier sous forme doc

In [41]:
# =============================================
#   CODE FINAL – À COPIER-COLLER ET EXÉCUTER
# =============================================

from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE
import datetime
from google.colab import files

def create_professional_eda_quality_doc(eda_txt_path, quality_txt_path, output_docx="Rapport_Scouts_EDA_Qualite.docx"):
    doc = Document()

    # ───────────── Styles personnalisés ─────────────
    style_title = doc.styles.add_style('CustomTitle', WD_STYLE_TYPE.PARAGRAPH)
    style_title.font.name = 'Arial'
    style_title.font.size = Pt(22)
    style_title.font.bold = True
    style_title.font.color.rgb = RGBColor(0, 51, 102)  # Bleu foncé

    style_section = doc.styles.add_style('SectionHeading', WD_STYLE_TYPE.PARAGRAPH)
    style_section.font.name = 'Arial'
    style_section.font.size = Pt(16)
    style_section.font.bold = True
    style_section.font.color.rgb = RGBColor(0, 102, 204)  # Bleu clair

    style_good = doc.styles.add_style('Good', WD_STYLE_TYPE.PARAGRAPH)
    style_good.font.color.rgb = RGBColor(0, 128, 0)      # Vert

    style_warning = doc.styles.add_style('Warning', WD_STYLE_TYPE.PARAGRAPH)
    style_warning.font.color.rgb = RGBColor(204, 102, 0)  # Orange

    style_critical = doc.styles.add_style('Critical', WD_STYLE_TYPE.PARAGRAPH)
    style_critical.font.color.rgb = RGBColor(204, 0, 0)   # Rouge

    # ───────────── Page de garde ─────────────
    title = doc.add_paragraph("RAPPORT D'ANALYSE EXPLORATOIRE ET DE QUALITÉ", style='CustomTitle')
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph("Données Scouts – Multi-fichiers", style='Heading 2').alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph(f"Généré le {datetime.datetime.now().strftime('%d %B %Y à %H:%M')}", style='Normal').alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_page_break()

    # ───────────── Analyse Exploratoire (EDA) ─────────────
    doc.add_heading("1. Analyse Exploratoire (EDA)", level=1)

    with open(eda_txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    current_file = ""
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if "FILE:" in line:
            current_file = line.replace("FILE:", "").strip()
            p = doc.add_paragraph(current_file, style='SectionHeading')

        elif line.startswith("Dimensions:") or line.startswith("Rows:") or line.startswith("Columns:"):
            doc.add_paragraph(line, style='Normal')

        elif "Quality Score:" in line or "Completeness:" in line:
            p = doc.add_paragraph(line, style='Normal')
            if "100.0%" in line or "EXCELLENT" in line:
                p.style = 'Good'
            elif "90" in line or "GOOD" in line:
                p.style = 'Warning'
            else:
                p.style = 'Critical'

        elif line.startswith("Column:"):
            doc.add_paragraph(line, style='Heading 3')

        elif "Statistics:" in line:
            doc.add_paragraph(line, style='Normal')

    doc.add_page_break()

    # ───────────── Qualité des données ─────────────
    doc.add_heading("2. Évaluation de la Qualité des Données", level=1)

    with open(quality_txt_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if "FILE:" in line:
            current_file = line.replace("FILE:", "").strip()
            p = doc.add_paragraph(current_file, style='SectionHeading')

        elif "EXCELLENT" in line:
            p = doc.add_paragraph(line, style='Good')
        elif "GOOD" in line or "FAIR" in line:
            p = doc.add_paragraph(line, style='Warning')
        elif "POOR" in line or "NEEDS IMPROVEMENT" in line:
            p = doc.add_paragraph(line, style='Critical')
        elif line.startswith("✅"):
            p = doc.add_paragraph(line, style='Good')
        elif line.startswith("⚠️") or line.startswith("❌"):
            p = doc.add_paragraph(line, style='Warning')
        else:
            doc.add_paragraph(line, style='Normal')

    # ───────────── Sauvegarde et téléchargement ─────────────
    doc.save(output_docx)
    print(f"\nFichier Word généré avec succès : {output_docx}")
    files.download(output_docx)

# =============================================
#   REMPLACE ICI PAR TES VRAIS NOMS DE FICHIERS
# =============================================

# Exemple (remplace par ce que tu vois dans ta sortie) :
eda_txt_path     = "EDA_Report_20260201_2000.txt"  # <--- Mettez le bon nom ici
quality_txt_path = "Quality_Report_20260201_2000.txt" # ← CHANGE ÇA

# Exécute la fonction
create_professional_eda_quality_doc(
    eda_txt_path=eda_txt_path,
    quality_txt_path=quality_txt_path
)


Fichier Word généré avec succès : Rapport_Scouts_EDA_Qualite.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
SEPARATION

In [42]:
# =========================================================
#  VERSION 2 DOCUMENTS SÉPARÉS (EDA & QUALITÉ)
# =========================================================

def create_two_separate_reports(eda_txt_path, quality_txt_path):

    # --- FONCTION INTERNE POUR APPLIQUER LES STYLES ---
    def setup_styles(doc):
        style_section = doc.styles.add_style('SectionHeading', WD_STYLE_TYPE.PARAGRAPH)
        style_section.font.name = 'Arial'
        style_section.font.size = Pt(16)
        style_section.font.bold = True
        style_section.font.color.rgb = RGBColor(0, 102, 204)

        doc.styles.add_style('Good', WD_STYLE_TYPE.PARAGRAPH).font.color.rgb = RGBColor(0, 128, 0)
        doc.styles.add_style('Warning', WD_STYLE_TYPE.PARAGRAPH).font.color.rgb = RGBColor(204, 102, 0)
        doc.styles.add_style('Critical', WD_STYLE_TYPE.PARAGRAPH).font.color.rgb = RGBColor(204, 0, 0)

    # 1. GÉNÉRATION DU RAPPORT EDA
    doc_eda = Document()
    setup_styles(doc_eda)
    doc_eda.add_heading("RAPPORT D'ANALYSE EXPLORATOIRE (EDA)", level=0)

    with open(eda_txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if "FILE:" in line:
                doc_eda.add_paragraph(line.replace("FILE:", "").strip(), style='SectionHeading')
            else:
                doc_eda.add_paragraph(line)

    name_eda = "Rapport_EDA_Final.docx"
    doc_eda.save(name_eda)
    print(f"✅ Fichier EDA prêt : {name_eda}")

    # 2. GÉNÉRATION DU RAPPORT QUALITÉ
    doc_qual = Document()
    setup_styles(doc_qual)
    doc_qual.add_heading("RAPPORT DE QUALITÉ DES DONNÉES", level=0)

    with open(quality_txt_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if "FILE:" in line:
                doc_qual.add_paragraph(line.replace("FILE:", "").strip(), style='SectionHeading')
            elif "EXCELLENT" in line or "✅" in line:
                doc_qual.add_paragraph(line, style='Good')
            elif "POOR" in line or "❌" in line:
                doc_qual.add_paragraph(line, style='Critical')
            else:
                doc_qual.add_paragraph(line)

    name_qual = "Rapport_Qualite_Final.docx"
    doc_qual.save(name_qual)
    print(f"✅ Fichier Qualité prêt : {name_qual}")

    # Téléchargement automatique des deux fichiers
    files.download(name_eda)
    files.download(name_qual)

# EXÉCUTION (Utilise les variables automatiques pour éviter l'erreur de fichier)
create_two_separate_reports(eda_txt, quality_txt)

✅ Fichier EDA prêt : Rapport_EDA_Final.docx
✅ Fichier Qualité prêt : Rapport_Qualite_Final.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Graphiquement

In [44]:
import os
from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.style import WD_STYLE_TYPE
from google.colab import files

def generate_final_scouts_reports(eda_path, quality_path, list_of_images=None):

    # --- CONFIGURATION DES STYLES ---
    def apply_styles(doc):
        # Style pour les titres de fichiers
        if 'FileHeading' not in doc.styles:
            style = doc.styles.add_style('FileHeading', WD_STYLE_TYPE.PARAGRAPH)
            style.font.name = 'Arial'
            style.font.size = Pt(14)
            style.font.bold = True
            style.font.color.rgb = RGBColor(0, 51, 153)

        # Styles de couleurs
        if 'Success' not in doc.styles:
            doc.styles.add_style('Success', WD_STYLE_TYPE.PARAGRAPH).font.color.rgb = RGBColor(0, 128, 0)
        if 'Danger' not in doc.styles:
            doc.styles.add_style('Danger', WD_STYLE_TYPE.PARAGRAPH).font.color.rgb = RGBColor(200, 0, 0)

    # ==========================================
    # 1. GÉNÉRATION DU RAPPORT EDA (AVEC GRAPHES)
    # ==========================================
    doc_eda = Document()
    apply_styles(doc_eda)

    title_eda = doc_eda.add_heading("RAPPORT D'ANALYSE EXPLORATOIRE (EDA)", level=0)
    title_eda.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Insertion du texte de l'EDA
    if os.path.exists(eda_path):
        with open(eda_path, 'r', encoding='utf-8') as f:
            for line in f:
                clean_line = line.strip()
                if "FILE:" in clean_line:
                    doc_eda.add_paragraph(clean_line, style='FileHeading')
                else:
                    doc_eda.add_paragraph(clean_line)

    # INSERTION DES GRAPHIQUES (S'il y en a)
    if list_of_images:
        doc_eda.add_page_break()
        doc_eda.add_heading("VISUALISATIONS ET GRAPHIQUES", level=1)
        for img in list_of_images:
            if os.path.exists(img):
                doc_eda.add_paragraph(f"\nVisualisation : {img}", style='FileHeading')
                doc_eda.add_picture(img, width=Inches(5.5))
                doc_eda.add_paragraph("-" * 50)

    eda_filename = "Rapport_EDA_Final.docx"
    doc_eda.save(eda_filename)

    # ==========================================
    # 2. GÉNÉRATION DU RAPPORT QUALITÉ
    # ==========================================
    doc_qual = Document()
    apply_styles(doc_qual)

    title_qual = doc_qual.add_heading("RAPPORT DE QUALITÉ DES DONNÉES", level=0)
    title_qual.alignment = WD_ALIGN_PARAGRAPH.CENTER

    if os.path.exists(quality_path):
        with open(quality_path, 'r', encoding='utf-8') as f:
            for line in f:
                clean_line = line.strip()
                p = doc_qual.add_paragraph(clean_line)
                if "EXCELLENT" in clean_line or "✅" in clean_line:
                    p.style = 'Success'
                elif "POOR" in clean_line or "❌" in clean_line:
                    p.style = 'Danger'

    qual_filename = "Rapport_Qualite_Final.docx"
    doc_qual.save(qual_filename)

    # --- TÉLÉCHARGEMENT ---
    print("🚀 Préparation du téléchargement des deux rapports...")
    files.download(eda_filename)
    files.download(qual_filename)

# ==========================================
# LANCEMENT AUTOMATIQUE
# ==========================================

# 1. On récupère la liste des images .png créées dans le dossier actuel
import glob
mes_images = glob.glob("*.png")

# 2. On utilise les variables eda_txt et quality_txt définies plus haut dans ton notebook
try:
    generate_final_scouts_reports(
        eda_path=eda_txt,
        quality_path=quality_txt,
        list_of_images=mes_images
    )
except NameError:
    print("❌ Erreur : Les variables eda_txt ou quality_txt n'existent pas.")
    print("Assure-toi d'avoir exécuté la cellule qui génère les fichiers texte juste avant.")



🚀 Préparation du téléchargement des deux rapports...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ValueError: Could not interpret value `votre_colonne` for `x`. An entry with this name does not appear in `data`.